# Travel-time tomography and FWI

**Level 2 — Applied analysis**  
**Prerequisites:** Level 1 and arrays/logarithms; 35 min.

## Objectives

Trace the evidence from source contract to result. State units and assumptions, reproduce the calculation, and identify which inference the evidence cannot support.

## Setup

Uses repository modules and committed records through a discovered root; it downloads and mutates nothing. Install exact pins: `pip install numpy==2.2.6 matplotlib==3.11.2`

## Data

STEAD held-out traces and replay receipts are committed; distinguish measured counts, synthetic FWI, analyst labels and browser assets.

## Method

For source `s=(xs,zs)` and receiver `r=(xr,zr)`, a homogeneous control is `t=||r−s||/v` (m divided by m·s⁻¹). Tomographic sensitivity is approximately `δt=−∫ray δv/v² ds`; ray coverage and velocity-depth trade-off limit resolution. Koenigsee raw bytes are not present, so do not treat the upstream example as local data. FWI minimizes a waveform mismatch such as `½||Wd(F(m)−d)||²+regularization`, with velocity in m/s and wave-equation forward operator F. Adjoint gradients are standard; cycle skipping occurs when phase mismatch lies outside the starting model’s basin. Inspect the actual synthetic FWI_LAYERED truth, L2/multiscale models and active/withheld WRMS, on physical distance/depth axes. This supports a controlled comparison only, not field validation.

At 100 samples/s, Nyquist is 50 Hz; instrument response and preprocessing set usable bandwidth. Tomographic rays depend on current velocity; FWI uses full waveform phase and amplitude. Keep held-out traces/shots outside fitting. Lower training residual may coexist with worse withheld WRMS or cycle-skipped structure.

## Results

Read the physical axes and units; distinguish data fit, error against synthetic truth and agreement with analyst labels. Recomputed values must agree with the serialized receipt. A plotted model alone does not establish uniqueness, field transfer or resolution.

### Diagnostic 1

Inspect the committed case record and provenance.

In [ ]:
from pathlib import Path
from importlib.metadata import version
print("Runtime versions:", {name: version(name) for name in ['numpy', 'matplotlib']})
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"data").is_dir() and (p/"data-pipeline").is_dir())
# Inspect the committed synthetic FWI replay and physical grid.
import json
from pathlib import Path
replay=json.loads((root/"data/derived/v2/fwi-replay.json").read_text(encoding="utf-8")); case=json.loads((root/"data/derived/v2/FWI_LAYERED/regularization.json").read_text(encoding="utf-8")); rows=[r for r in replay["records"] if r["case"]=="FWI_LAYERED" and r["variant"]=="reference"]
print("synthetic",case["id"],"grid [x,z]",case["grid"]["shape"],"spacing m",case["grid"]["spacing"],"dt s",case["dt"]); print([(r["method"],r["velocity_rmse"],r["active_wrms"],r["withheld_wrms"]) for r in rows])

### Diagnostic 2

Compute the declared comparison and plot it with physical units.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Compare saved truth, L2 and multiscale velocity fields with physical x/depth axes.
import matplotlib.pyplot as plt
nx,nz=case["grid"]["shape"]; dx,dz=case["grid"]["spacing"]; x=np.arange(nx)*dx; z=np.arange(nz)*dz; truth=np.asarray(case["truth"]).reshape(nx,nz).T; fig,ax=plt.subplots(1,3,figsize=(11,3.5),constrained_layout=True)
for a,v,title in zip(ax,[truth]+[np.asarray(case["methods"][k]["model"]).reshape(nx,nz).T for k in ("fwi-l2","fwi-multiscale")],["truth","L2","multiscale"]): im=a.imshow(v,origin="upper",aspect="auto",extent=[x.min(),x.max(),z.max(),z.min()]); a.set(title=title,xlabel="Distance (m)",ylabel="Depth (m)")
fig.colorbar(im,ax=ax,label="Velocity (m/s)"); plt.show(); print("velocity RMSE (m/s), active/withheld WRMS:",[(r["method"],r["velocity_rmse"],r["active_wrms"],r["withheld_wrms"]) for r in rows])

## Summary

Conclusion is conditional on geometry, acquisition, noise model, split, parameterization and source rights. Preserve negative cases and name the next observation or control needed.

## References

- [https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html)
- [https://www.pygimli.org/_examples_auto/2_seismics/plot_04_koenigsee/](https://www.pygimli.org/_examples_auto/2_seismics/plot_04_koenigsee/)
- [https://www.devitoproject.org/examples/seismic/tutorials/03_fwi.html](https://www.devitoproject.org/examples/seismic/tutorials/03_fwi.html)